# SatDetection — Colab Training Notebook
**Multi-task U-Net + ResNet-50 for building and road segmentation**

CSCI 4366/6366 — Neural Networks and Deep Learning, Spring 2026  
Team: Rufai Yakubu, Aaron Tyler

---
**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Check GPU

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install Dependencies

In [ ]:
%%capture
!pip install segmentation-models-pytorch albumentations rasterio wandb pyyaml tqdm

## 3. Mount Google Drive
All data, checkpoints, and outputs persist on Drive across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR       = '/content/drive/MyDrive/SatDetection'
DATA_DIR       = os.path.join(BASE_DIR, 'data')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
VISUALS_DIR    = os.path.join(BASE_DIR, 'outputs/visuals')

for d in [
    DATA_DIR, CHECKPOINT_DIR, VISUALS_DIR,
    os.path.join(DATA_DIR, 'landcoverai/raw/images'),
    os.path.join(DATA_DIR, 'landcoverai/raw/masks'),
    os.path.join(DATA_DIR, 'landcoverai/tiles/images'),
    os.path.join(DATA_DIR, 'landcoverai/tiles/masks'),
]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted and directories ready.')

## 4. Clone Project Repository
Replace the URL with your GitHub repo URL.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/YOUR_USERNAME/SatDetection.git'  # <-- update this

if not os.path.exists('/content/SatDetection'):
    !git clone $REPO_URL /content/SatDetection
else:
    !git -C /content/SatDetection pull

sys.path.insert(0, '/content/SatDetection/src')
print('Repo ready.')

## 5. W&B Login

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted

## 6. Data Preparation — Tile LandCover.ai

**One-time step.** Skip this cell if tiles already exist on Drive.

Download LandCover.ai from https://landcover.ai/ and upload to Drive:
```
MyDrive/SatDetection/data/landcoverai/raw/
  images/   *.tif
  masks/    *.tif   (0=background, 1=building, 2=road)
```

In [ ]:
import glob
from utils import tile_all

LC_IMAGE_DIR = os.path.join(DATA_DIR, 'landcoverai/raw/images')
LC_MASK_DIR  = os.path.join(DATA_DIR, 'landcoverai/raw/masks')
LC_TILES_OUT = os.path.join(DATA_DIR, 'landcoverai/tiles')

existing = len(glob.glob(os.path.join(LC_TILES_OUT, 'images/*.npy')))
if existing > 0:
    print(f'Tiles already exist: {existing} tiles. Skipping tiling.')
else:
    total = tile_all(LC_IMAGE_DIR, LC_MASK_DIR, LC_TILES_OUT, tile_size=256, stride=256)
    print(f'Tiling complete. Total tiles: {total}')

In [ ]:
# Verify tile counts
n_images = len(glob.glob(os.path.join(DATA_DIR, 'landcoverai/tiles/images/*.npy')))
n_masks  = len(glob.glob(os.path.join(DATA_DIR, 'landcoverai/tiles/masks/*.npy')))
print(f'Image tiles : {n_images}')
print(f'Mask tiles  : {n_masks}')
assert n_images == n_masks, 'Mismatch between image and mask tile counts!'

## 7. Training Config

In [ ]:
BASE_CONFIG = {
    'landcoverai_root':        os.path.join(DATA_DIR, 'landcoverai/tiles'),
    'checkpoint_dir':          CHECKPOINT_DIR,
    'visuals_dir':             VISUALS_DIR,
    'wandb_project':           'SatDetection',
    'lr':                      0.0001,
    'epochs':                  50,
    'batch_size':              16,   # T4 can handle 16
    'num_workers':             2,
    'early_stopping_patience': 5,
    'w_building':              1.0,
    'w_road':                  2.0,
}

EXPERIMENTS = [
    {'pretrained': False, 'name': 'Model A — Random Init'},
    {'pretrained': True,  'name': 'Model B — ImageNet Pretrained'},
]

print('Experiment plan:')
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f'  Run {i}: {exp["name"]}')

## 8. Training
Run each cell separately. Checkpoints auto-save to Drive after every best epoch.

In [ ]:
from train import train

def run_experiment(exp_index):
    exp    = EXPERIMENTS[exp_index - 1]
    config = {**BASE_CONFIG, 'pretrained': exp['pretrained']}
    print(f"\n{'='*60}")
    print(f"Run {exp_index}: {exp['name']}")
    print(f"{'='*60}")
    checkpoint = train(config)
    print(f"Checkpoint: {checkpoint}")
    return checkpoint

In [ ]:
# Run 1 — Model A (random init, baseline)
ckpt_1 = run_experiment(1)

In [ ]:
# Run 2 — Model B (ImageNet pretrained)
ckpt_2 = run_experiment(2)

## 9. Evaluation

In [ ]:
from evaluate import evaluate

def eval_experiment(exp_index, checkpoint_path):
    exp    = EXPERIMENTS[exp_index - 1]
    config = {**BASE_CONFIG, 'pretrained': exp['pretrained']}
    print(f"\n{'='*60}")
    print(f"Evaluating Run {exp_index}: {exp['name']}")
    print(f"{'='*60}")
    return evaluate(config, checkpoint_path, save_visuals=True, n_visuals=5)

In [ ]:
results_1 = eval_experiment(1, ckpt_1)

In [ ]:
results_2 = eval_experiment(2, ckpt_2)

## 10. Results Summary

In [ ]:
import pandas as pd

rows = []
for exp, results in [
    (EXPERIMENTS[0], results_1),
    (EXPERIMENTS[1], results_2),
]:
    rows.append({
        'Model':        exp['name'],
        'IoU Building': f"{results['iou_building']:.4f}",
        'IoU Road':     f"{results['iou_road']:.4f}",
        'Mean IoU':     f"{(results['iou_building'] + results['iou_road']) / 2:.4f}",
        'F1 Building':  f"{results['f1_building']:.4f}",
        'F1 Road':      f"{results['f1_road']:.4f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))